# 내 계좌 6대 위기 스트레스 테스트 · 폭락장 대응 시뮬레이터 (Google Colab)

**목적:** 폭락장이 왔을 때 당황해서 저점 매도하지 않도록, **내 A+B+C 포트폴리오를 6가지 역사적·가상 위기 시나리오에 미리 넣어보고**
1. **각 위기(2008 금융위기 · 2020 코로나 · 2022 고금리 · 반도체 -50% · 금융배당 -30% · 환율 급락)에서 내 계좌가 최대 몇 % 빠지는지(원화 기준)**
2. **주가 폭락 중에도 연간 배당 현금흐름은 몇 %나 지켜지는지**
3. **폭락장에서 「그냥 버티기 vs 매월 적립 지속 vs 하락장 자동 리밸런싱」의 원금 회복 기간 차이**
4. **내 주요 보유 자산(A 배당 · B 반도체 · C 미국/달러) 간 실제 10년 상관관계 히트맵**
을 **API 키 없이** 분석하고, **인쇄해 두고 쓸 수 있는 5페이지 A4 PDF 리포트(폭락장 행동 매뉴얼 포함)**를 Google Drive에 저장합니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트(NanumGothic) 및 PDF 엔진 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, io, math, base64, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint", "pyyaml"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    import yaml
except ImportError:
    yaml = None

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")

In [ ]:
# ==============================================================================
# 2. 포트폴리오 비중 설정 · Google Drive 연결 (비공개 YAML 자동 연동)
# ==============================================================================
STRESS_CONFIG = {
    # 기준 자산 & 월 적립금 (회복 기간 시뮬레이션용, 단위: 만 원)
    "INIT_ASSET_MAN": 10000,         # 기준 자산 (1억 원)
    "MONTHLY_DCA_MAN": 200,          # 폭락장 월 적립금 (200만 원)
    "FIN_SHARE_IN_A": 0.45,          # A층 내 금융주(은행·보험·지주) 비중 (기본 45%)

    # 비교할 4가지 포트폴리오
    "PORTFOLIOS": {
        "내 현행 50/20/30":    {"A": 0.50, "B": 0.20, "C": 0.30},
        "미국강화 40/15/45":   {"A": 0.40, "B": 0.15, "C": 0.45},
        "국내배당 A 100%":     {"A": 1.00, "B": 0.00, "C": 0.00},
        "반도체·AI B 100%":    {"A": 0.00, "B": 1.00, "C": 0.00},
    },
    "PRIMARY_NAME": "내 현행 50/20/30",

    # 상관관계 히트맵에 포함할 대표 자산군 (공개 시장 대표 종목/ETF)
    "CORR_ASSETS": [
        ("KB금융 (A·은행)",   "105560.KS"),
        ("삼성화재우 (A·보험)", "000815.KS"),
        ("현대차3우B (A·제조)", "005389.KS"),
        ("삼성전자우 (B·반도체)","005935.KS"),
        ("미국 S&P500 (C·달러)","SPY"),
        ("미국 나스닥 (C·기술)", "QQQ"),
        ("원/달러 환율",       "KRW=X"),
    ],

    # Google Drive 저장 경로
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_stress_test",
    "PORTFOLIO_YAML": "/content/drive/MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml",
}

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
if IN_COLAB and STRESS_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(STRESS_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 리포트 저장 경로: {DRIVE_REPORT_DIR}")

        pf_p = Path(STRESS_CONFIG["PORTFOLIO_YAML"])
        if yaml is not None and pf_p.exists():
            pf = yaml.safe_load(pf_p.read_text(encoding="utf-8")) or {}
            bt = pf.get("basket_targets") or {}
            if all(k in bt for k in ("A", "B", "C")):
                wA = float(bt["A"].get("target", 0.50))
                wB = float(bt["B"].get("target", 0.20))
                wC = float(bt["C"].get("target", 0.30))
                s = wA + wB + wC
                if s > 0:
                    STRESS_CONFIG["PORTFOLIOS"]["내 현행 50/20/30"] = {"A": wA/s, "B": wB/s, "C": wC/s}
                    print(f"[비공개 YAML 연동] 목표 비중 A={wA/s:.0%} / B={wB/s:.0%} / C={wC/s:.0%}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 저장 후 Cell 5에서 재시도합니다 ({type(e).__name__}: {e})")

print(f"[설정 완료] 비교 포트폴리오 {len(STRESS_CONFIG['PORTFOLIOS'])}개 | 기준자산 {STRESS_CONFIG['INIT_ASSET_MAN']:,}만 원 | 월 적립 {STRESS_CONFIG['MONTHLY_DCA_MAN']:,}만 원")

In [ ]:
# ==============================================================================
# 3. 10년 일별/월별 시세 수집 (실제 상관관계 히트맵 & 과거 위기 검증)
# ==============================================================================
KST = timezone(timedelta(hours=9))
as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-stress-test)"}

def fetch_yf_weekly(symbol: str) -> pd.Series:
    try:
        r = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
            params={"range": "10y", "interval": "1wk"},
            headers=_HEADERS,
            timeout=15,
        )
        r.raise_for_status()
        res = r.json()["chart"]["result"][0]
        ts = res["timestamp"]
        adj = res["indicators"].get("adjclose", [{}])[0].get("adjclose")
        vals = adj if adj is not None else res["indicators"]["quote"][0]["close"]
        idx = [pd.Timestamp(datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d")) for t in ts]
        s = pd.Series(vals, index=idx, dtype=float).dropna()
        s = s[~s.index.duplicated(keep="last")].sort_index()
        if symbol == "KRW=X":
            s = s[(s >= 800.0) & (s <= 2000.0)]
        return s.resample("W-FRI").last().dropna()
    except Exception:
        return pd.Series(dtype=float)

print("[데이터 수집] Yahoo Finance 10년 주봉 시세로 자산군 실제 상관관계 계산 중...")
WK_SERIES = {}
for lbl, sym in STRESS_CONFIG["CORR_ASSETS"]:
    s = fetch_yf_weekly(sym)
    if len(s) >= 100:
        WK_SERIES[lbl] = s

# 미국 지수(SPY, QQQ)는 한국 투자자 관점의 원화 환산 수익률도 함께 확인
if "원/달러 환율" in WK_SERIES:
    fx_s = WK_SERIES["원/달러 환율"]
    for usd_lbl in ("미국 S&P500 (C·달러)", "미국 나스닥 (C·기술)"):
        if usd_lbl in WK_SERIES:
            aligned = pd.concat({"p": WK_SERIES[usd_lbl], "fx": fx_s}, axis=1).dropna()
            if len(aligned) >= 100:
                WK_SERIES[usd_lbl.replace("·달러", "·원화").replace("·기술", "·원화")] = aligned["p"] * aligned["fx"]
                del WK_SERIES[usd_lbl]

RET_DF = pd.DataFrame({k: v.pct_change() for k, v in WK_SERIES.items()}).dropna()
CORR_DF = RET_DF.corr()
print(f"[상관관계 완료] 유효 자산 {len(CORR_DF)}개 · 공통 주간 표본 {len(RET_DF)}주 (~{len(RET_DF)/52:.1f}년)")

In [ ]:
# ==============================================================================
# 4. 6대 역사적·가상 위기 스트레스 테스트 + 폭락장 대응(방치 vs 적립 vs 리밸런싱) 시뮬레이션
# ==============================================================================
# 각 시나리오는 역사적 실측치(2008·2020·2022) 및 핵심 리스크 가정 기반 (원화 기준 수익률)
# - A_nonfin: 비금융 고배당주 / A_fin: 금융 배당주 / B: 반도체·AI / C_usd: 미국지수(달러) / fx: 원/달러 환율 변화
# - div_A_chg / div_B_chg / div_C_chg: 위기 시 연간 배당금 변화율
SCENARIOS = [
    {
        "name": "① 2008 글로벌 금융위기",
        "desc": "글로벌 주식 -45~55% 폭락, 단 원/달러 +35% 급등으로 미국 C층 환율 방어",
        "A_nonfin": -0.42, "A_fin": -0.52, "B": -0.48, "C_usd": -0.45, "fx": +0.35,
        "div_A": -0.18, "div_B": -0.25, "div_C": -0.15,
    },
    {
        "name": "② 2020 코로나 쇼크",
        "desc": "단기 2개월 만에 전 시장 -30~35% 급락 + 원/달러 +10% 상승 후 빠른 반등",
        "A_nonfin": -0.32, "A_fin": -0.36, "B": -0.30, "C_usd": -0.32, "fx": +0.10,
        "div_A": -0.05, "div_B":  0.00, "div_C":  0.00,
    },
    {
        "name": "③ 2022 고금리·인플레 침체",
        "desc": "반도체·나스닥 -30~35% 급락, 고배당·금융주 및 강달러(+14%) 상대적 방어",
        "A_nonfin": -0.15, "A_fin": -0.08, "B": -0.34, "C_usd": -0.24, "fx": +0.14,
        "div_A": +0.05, "div_B":  0.00, "div_C": +0.04,
    },
    {
        "name": "④ 반도체·AI 단독 -50% 쇼크",
        "desc": "AI 설비투자 피크아웃으로 반도체 B층 반토막, 미국 기술주 동반 -25%",
        "A_nonfin": -0.10, "A_fin": -0.08, "B": -0.50, "C_usd": -0.22, "fx": +0.08,
        "div_A":  0.00, "div_B": -0.30, "div_C":  0.00,
    },
    {
        "name": "⑤ 금융주 배당규제·침체 쇼크",
        "desc": "국내 은행·보험 건전성 규제로 금융주 -30% 하락 및 금융 배당 30% 감액",
        "A_nonfin": -0.08, "A_fin": -0.30, "B": -0.05, "C_usd":  0.00, "fx": +0.03,
        "div_A": -0.15, "div_B":  0.00, "div_C": +0.03,
    },
    {
        "name": "⑥ 원화 초강세(환율 -15%) 쇼크",
        "desc": "원/달러 1,400원 → 1,190원 급락 시 미국 ETF(C층) 원화 환산 수익률 타격",
        "A_nonfin": +0.03, "A_fin": +0.05, "B": -0.04, "C_usd":  0.00, "fx": -0.15,
        "div_A":  0.00, "div_B":  0.00, "div_C": -0.15,
    },
]

fin_s = STRESS_CONFIG["FIN_SHARE_IN_A"]
INIT_M = STRESS_CONFIG["INIT_ASSET_MAN"]
LAYER_YLD = {"A": 0.048, "B": 0.020, "C": 0.013}

STRESS_ROWS = []
MDD_MATRIX = {}  # portfolio -> list of crisis returns (%)

for pname, w in STRESS_CONFIG["PORTFOLIOS"].items():
    MDD_MATRIX[pname] = []
    for sc in SCENARIOS:
        r_a = (1.0 - fin_s) * sc["A_nonfin"] + fin_s * sc["A_fin"]
        r_b = sc["B"]
        r_c = (1.0 + sc["C_usd"]) * (1.0 + sc["fx"]) - 1.0  # 원화 환산
        port_ret = w["A"] * r_a + w["B"] * r_b + w["C"] * r_c
        MDD_MATRIX[pname].append(port_ret * 100.0)

        if pname == STRESS_CONFIG["PRIMARY_NAME"]:
            # 배당금 방어율 계산
            base_div = w["A"] * LAYER_YLD["A"] + w["B"] * LAYER_YLD["B"] + w["C"] * LAYER_YLD["C"]
            aft_div = (
                w["A"] * LAYER_YLD["A"] * (1.0 + sc["div_A"])
                + w["B"] * LAYER_YLD["B"] * (1.0 + sc["div_B"])
                + w["C"] * LAYER_YLD["C"] * (1.0 + sc["div_C"])
            )
            div_keep = (aft_div / base_div) * 100.0 if base_div > 0 else 100.0
            loss_man = INIT_M * port_ret
            # 원화 환산 C층 수익률 (환율 방어 효과 표시)
            STRESS_ROWS.append({
                "위기 시나리오": sc["name"],
                "내 현행(50/20/30)": f"{port_ret*100:+.1f}%",
                "평가손익(1억 기준)": f"{loss_man:+,.0f}만 원",
                "미국강화(40/15/45)": f"{(0.40*r_a + 0.15*r_b + 0.45*r_c)*100:+.1f}%",
                "반도체 단독(B100)": f"{r_b*100:+.1f}%",
                "C층 원화수익(환율반영)": f"{r_c*100:+.1f}%",
                "배당 현금흐름 방어율": f"{div_keep:.0f}%",
            })

STRESS_DF = pd.DataFrame(STRESS_ROWS)

# ------------------------------------------------------------------------------
# 폭락장 대응 행동 시뮬레이션 (가상 대형 위기: 6개월간 -35% 급락 -> 30개월간 점진 회복)
#   비교 3가지:
#   1) 거치식 방치 (추가매수 X, 리밸런싱 X)
#   2) 매월 적립 지속 (월 200만 원 정액 매수, 배당 재투자)
#   3) 월 적립 + 하락장 리밸런싱 (덜 빠진 C/A에서 많이 빠진 층으로 비중 복원 -> 반등 탄력 극대화)
# ------------------------------------------------------------------------------
months_sim = 48
t_idx = np.arange(0, months_sim + 1)

# 3개 층의 현실적 3단계 경로 (0~6개월 급락: A -28%, B -48%, C(원화) -10%[환율급등 방어] -> 7~18개월 반등: A +35%, B +75%, C +15%[환율정상화] -> 19~48개월 정상성장)
rA_m = np.array([((1 - 0.28)**(1/6) - 1)]*6 + [((1.35)**(1/12) - 1)]*12 + [((1.09)**(1/12) - 1)]*30)
rB_m = np.array([((1 - 0.48)**(1/6) - 1)]*6 + [((1.75)**(1/12) - 1)]*12 + [((1.12)**(1/12) - 1)]*30)
rC_m = np.array([((1 - 0.10)**(1/6) - 1)]*6 + [((1.15)**(1/12) - 1)]*12 + [((1.10)**(1/12) - 1)]*30)

w_pri = STRESS_CONFIG["PORTFOLIOS"][STRESS_CONFIG["PRIMARY_NAME"]]
w_vec = np.array([w_pri["A"], w_pri["B"], w_pri["C"]], dtype=float)
dca_m = STRESS_CONFIG["MONTHLY_DCA_MAN"]
div_m = np.array([LAYER_YLD["A"], LAYER_YLD["B"], LAYER_YLD["C"]]) * (1.0 - 0.154) / 12.0

def sim_crash_response(mode: str) -> tuple[np.ndarray, np.ndarray]:
    vals = np.zeros((months_sim + 1, 3), dtype=float)
    vals[0] = INIT_M * w_vec
    cum_prin = np.full(months_sim + 1, float(INIT_M), dtype=float)
    for m in range(1, months_sim + 1):
        rm = np.array([rA_m[m-1], rB_m[m-1], rC_m[m-1]])
        cur = vals[m-1] * (1.0 + rm + div_m)
        if mode == "lump":
            vals[m] = cur
            cum_prin[m] = cum_prin[m-1]
        elif mode == "dca":
            vals[m] = cur + dca_m * w_vec
            cum_prin[m] = cum_prin[m-1] + dca_m
        elif mode == "dca_rebal":
            tot = cur.sum() + dca_m
            # 반기(6개월)마다 또는 비중 ±6%p 이탈 시 목표 비중으로 자동 리밸런싱 (저가 매수 효과)
            cur_w = cur / max(cur.sum(), 1.0)
            if m % 6 == 0 or np.max(np.abs(cur_w - w_vec)) >= 0.06:
                vals[m] = tot * w_vec
            else:
                deficit = np.maximum(0.0, tot * w_vec - cur)
                alloc = (deficit / deficit.sum()) if deficit.sum() > 0 else w_vec
                vals[m] = cur + dca_m * alloc
            cum_prin[m] = cum_prin[m-1] + dca_m
    tot_v = vals.sum(axis=1)
    ret_pct = (tot_v / cum_prin - 1.0) * 100.0
    return tot_v, ret_pct

V_LUMP, R_LUMP = sim_crash_response("lump")
V_DCA,  R_DCA  = sim_crash_response("dca")
V_REB,  R_REB  = sim_crash_response("dca_rebal")

def find_recovery_month(ret_arr: np.ndarray) -> int:
    hits = np.where(ret_arr[7:] >= 0.0)[0]
    return int(hits[0] + 7) if len(hits) else months_sim

REC_LUMP = find_recovery_month(R_LUMP)
REC_DCA  = find_recovery_month(R_DCA)
REC_REB  = find_recovery_month(R_REB)

print(f"■ [내 포트폴리오({STRESS_CONFIG['PRIMARY_NAME']}) 6대 위기 스트레스 테스트 ({as_of_str})]")
display(STRESS_DF)
print(f"\n■ [폭락장(-35%) 원금 회복 소요 기간] 거치식 방치 {REC_LUMP}개월 vs 월적립 지속 {REC_DCA}개월 vs 적립+리밸런싱 {REC_REB}개월")
print(f"   4년(48개월) 후 최종 자산: 적립+리밸런싱 {V_REB[-1]/10000:.2f}억 vs 단순 적립 {V_DCA[-1]/10000:.2f}억 (차이 {(V_REB[-1]-V_DCA[-1]):+,.0f}만 원)")

In [ ]:
# ==============================================================================
# 5. 고가독성 풀폭 차트 3종 + 5페이지 A4 PDF/HTML 리포트(폭락장 매뉴얼 포함) & Drive 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64_list: list[str]) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception as e:
        print(f"[안내] WeasyPrint 폴백({type(e).__name__}: {e}) -> Matplotlib PDF로 생성")
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            for b64 in chart_b64_list:
                img = mpimg.imread(io.BytesIO(base64.b64decode(b64)), format="png")
                fig_p = plt.figure(figsize=(11.69, 8.27))
                plt.imshow(img); plt.axis("off"); plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception:
        return False

# ------------------------------------------------------------------------------
# 차트 ① (풀폭 단독): 6대 위기 시나리오별 포트폴리오 최대 손실폭 비교
# ------------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(13.8, 6.0), facecolor="white")
sc_labels = [s["name"].replace(" ", "\n", 1) for s in SCENARIOS]
x = np.arange(len(SCENARIOS))
p_keys = list(STRESS_CONFIG["PORTFOLIOS"].keys())
p_cols = ["#0F766E", "#2563EB", "#D97706", "#DC2626"]
bw = 0.19

for idx_p, (pk, col) in enumerate(zip(p_keys, p_cols)):
    vals = MDD_MATRIX[pk]
    offs = (idx_p - 1.5) * bw
    bars = ax1.bar(x + offs, vals, width=bw, color=col, label=pk, zorder=3)
    for b, v in zip(bars, vals):
        y_pos = v - 1.2 if v < 0 else v + 0.8
        va = "top" if v < 0 else "bottom"
        ax1.text(b.get_x() + b.get_width()/2, y_pos, f"{v:+.0f}%", ha="center", va=va, fontsize=7.8, fontweight="bold", color=col)

ax1.axhline(0, color="#0F172A", lw=1.2, zorder=4)
ax1.set_xticks(x); ax1.set_xticklabels(sc_labels, fontsize=9.2, fontweight="bold")
ax1.set_ylabel("위기 시 원화 기준 수익률 / 낙폭 (%)", fontsize=10.2, fontweight="bold")
ax1.set_ylim(-58, 12)
ax1.set_title("① 6대 역사적·가상 위기 시나리오별 포트폴리오 방어력 비교 (원화 기준)",
              fontsize=13.2, fontweight="bold", color="#0F2922", loc="left", pad=26)
ax1.legend(loc="upper center", bbox_to_anchor=(0.5, 1.08), ncol=4, fontsize=9.0, frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1")
ax1.grid(True, axis="y", color="#E2E8F0", ls="--", lw=0.8)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
fig1.tight_layout()
b1 = io.BytesIO(); fig1.savefig(b1, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART1_B64 = base64.b64encode(b1.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig1)

# ------------------------------------------------------------------------------
# 차트 ② (풀폭 단독): 폭락장(-35%) 대응별 원금 회복 속도 & 4년 후 자산 비교
# ------------------------------------------------------------------------------
fig2, (ax2l, ax2r) = plt.subplots(1, 2, figsize=(14.0, 5.7), facecolor="white")
ax2l.axvspan(0, 6, color="#FEE2E2", alpha=0.55, label="폭락 구간 (0~6개월)")
ax2l.axhline(0, color="#0F172A", ls="--", lw=1.2)
ax2l.plot(t_idx, R_REB,  color="#0F766E", lw=2.8, label=f"월 적립 + 자동 리밸런싱 (원금회복 {REC_REB}개월)")
ax2l.plot(t_idx, R_DCA,  color="#2563EB", lw=2.2, ls="--", label=f"월 적립 지속 (원금회복 {REC_DCA}개월)")
ax2l.plot(t_idx, R_LUMP, color="#94A3B8", lw=2.0, ls=":", label=f"거치식 방치 (원금회복 {REC_LUMP}개월)")

for m_rec, col, yoff in [(REC_REB, "#0F766E", 10), (REC_LUMP, "#64748B", -20)]:
    ax2l.scatter([m_rec], [0], color=col, s=65, zorder=5)
    ax2l.annotate(f"{m_rec}개월 회복", xy=(m_rec, 0), xytext=(4, yoff), textcoords="offset points",
                  fontsize=8.6, fontweight="bold", color=col,
                  bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=col, lw=0.8))

ax2l.set_xlabel("경과 개월 수 (월)", fontsize=10.0, fontweight="bold")
ax2l.set_ylabel("누적 투자원금 대비 수익률 (%)", fontsize=10.0, fontweight="bold")
ax2l.set_title("②-A 폭락장 대응 방식별 원금 회복 속도 비교", fontsize=11.8, fontweight="bold", color="#0F2922", loc="left", pad=10)
ax2l.legend(loc="lower right", fontsize=8.4, frameon=True, facecolor="white")
ax2l.grid(True, color="#E2E8F0", ls="--", lw=0.8)
ax2l.spines["top"].set_visible(False); ax2l.spines["right"].set_visible(False)

# 우측: 48개월 후 총 평가자산 비교
ax2r.plot(t_idx, V_REB/10000, color="#0F766E", lw=2.8, label=f"적립 + 자동 리밸런싱 (4년 후 {V_REB[-1]/10000:.2f}억)")
ax2r.plot(t_idx, V_DCA/10000, color="#2563EB", lw=2.2, ls="--", label=f"단순 월 적립 (4년 후 {V_DCA[-1]/10000:.2f}억)")
ax2r.plot(t_idx, V_LUMP/10000, color="#94A3B8", lw=2.0, ls=":", label=f"거치식 방치 (4년 후 {V_LUMP[-1]/10000:.2f}억)")
diff_man = V_REB[-1] - V_DCA[-1]
ax2r.annotate(f"리밸런싱 추가효과\n{diff_man:+,.0f}만 원", xy=(48, V_REB[-1]/10000), xytext=(-95, -24),
              textcoords="offset points", fontsize=9.0, fontweight="bold", color="#0F766E",
              bbox=dict(boxstyle="round,pad=0.28", facecolor="#ECFDF5", edgecolor="#0F766E", lw=1.0),
              arrowprops=dict(arrowstyle="->", color="#0F766E", lw=1.1))
ax2r.set_xlabel("경과 개월 수 (월)", fontsize=10.0, fontweight="bold")
ax2r.set_ylabel("총 평가자산 (억 원)", fontsize=10.0, fontweight="bold")
ax2r.set_title("②-B 폭락장 이후 4년 누적 자산 성장 비교", fontsize=11.8, fontweight="bold", color="#0F2922", loc="left", pad=10)
ax2r.legend(loc="upper left", fontsize=8.4, frameon=True, facecolor="white")
ax2r.grid(True, color="#E2E8F0", ls="--", lw=0.8)
ax2r.spines["top"].set_visible(False); ax2r.spines["right"].set_visible(False)

fig2.tight_layout()
b2 = io.BytesIO(); fig2.savefig(b2, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART2_B64 = base64.b64encode(b2.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig2)

# ------------------------------------------------------------------------------
# 차트 ③ (풀폭 단독): 실제 10년 자산군 상관관계 히트맵 (왜 A+B+C+달러가 위험을 줄이는가?)
# ------------------------------------------------------------------------------
fig3, ax3 = plt.subplots(figsize=(11.8, 5.8), facecolor="white")
cmat = CORR_DF.values
clbls = list(CORR_DF.columns)
im = ax3.imshow(cmat, cmap="RdYlGn_r", vmin=-0.6, vmax=1.0, aspect="auto")
ax3.set_xticks(np.arange(len(clbls))); ax3.set_xticklabels(clbls, fontsize=9.0, fontweight="bold", rotation=20, ha="right")
ax3.set_yticks(np.arange(len(clbls))); ax3.set_yticklabels(clbls, fontsize=9.2, fontweight="bold")
for i in range(len(clbls)):
    for j in range(len(clbls)):
        val = cmat[i, j]
        txt_col = "white" if (val > 0.65 or val < -0.25) else "#0F172A"
        ax3.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=9.8, fontweight="bold", color=txt_col)
cbar = fig3.colorbar(im, ax=ax3, fraction=0.032, pad=0.03)
cbar.set_label("상관계수 (-1.0 완전 역방향 · 0.0 무관 · +1.0 동반 움직임)", fontsize=9.0)
ax3.set_title(f"③ 내 포트폴리오 핵심 자산군 10년 실측 주간 상관관계 히트맵 ({len(RET_DF)}주 기준)",
              fontsize=12.8, fontweight="bold", color="#0F2922", loc="left", pad=14)
fig3.tight_layout()
b3 = io.BytesIO(); fig3.savefig(b3, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART3_B64 = base64.b64encode(b3.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig3)

# ------------------------------------------------------------------------------
# 5페이지 A4 가로 고가독성 리포트 조립
# ------------------------------------------------------------------------------
def render_stress_tbl(df: pd.DataFrame) -> str:
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    rows = []
    for _, r in df.iterrows():
        tds = []
        for i, c in enumerate(df.columns):
            val = str(r[c])
            cls = "col-left" if i == 0 else ""
            if i == 1 and val.startswith("-"):
                cls += " txt-red"
            elif i == 6:
                cls += " txt-green"
            tds.append(f'<td class="{cls}">{escape(val)}</td>')
        rows.append("<tr>" + "".join(tds) + "</tr>")
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(rows)}</tbody></table>'

pri_worst = min(MDD_MATRIX[STRESS_CONFIG["PRIMARY_NAME"]])
semi_worst = min(MDD_MATRIX["반도체·AI B 100%"])
rec_saved = REC_LUMP - REC_REB
tbl_stress_html = render_stress_tbl(STRESS_DF)

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>내 계좌 6대 위기 스트레스 테스트 · 폭락장 매뉴얼 ({as_of_str})</title>
<style>
  @page {{ size: A4 landscape; margin: 9mm 10mm; }}
  body {{ margin:0; padding:10px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.3px; line-height:1.5; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:15px 20px; border-radius:10px; margin-bottom:10px; border-bottom:4px solid #D4A34A; }}
  .head h1 {{ margin:0 0 4px; font-size:17.5px; }}
  .head .sub {{ font-size:9.8px; opacity:0.92; }}
  .kpi-row {{ display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin-bottom:10px; }}
  .kpi {{ background:#fff; border:1px solid #E2E8F0; border-top:3px solid #0F766E; border-radius:8px; padding:9px 11px; }}
  .kpi .k-lbl {{ font-size:8.8px; color:#64748B; font-weight:700; }}
  .kpi .k-val {{ font-size:15.5px; font-weight:800; color:#0F2922; margin:2px 0; }}
  .kpi .k-sub {{ font-size:8.4px; color:#475569; }}
  .guide-box {{ background:#F0FDF9; border:1px solid #99F6E4; border-left:4px solid #0F766E; border-radius:8px; padding:9px 13px; margin-bottom:10px; font-size:9.3px; }}
  .guide-grid {{ display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:5px; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:11px 14px; margin-bottom:10px; break-inside:avoid; }}
  .card h2 {{ margin:0 0 8px; font-size:12.6px; color:#0F2922; border-bottom:1.5px solid #E2E8F0; padding-bottom:5px; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.3px; }}
  table.tbl th {{ background:#0F2922; color:#fff; padding:6.5px 6px; text-align:right; font-weight:700; }}
  table.tbl td {{ padding:6.5px 6px; border-bottom:1px solid #E2E8F0; text-align:right; font-weight:700; }}
  table.tbl td.col-left, table.tbl th:first-child {{ text-align:left; }}
  .txt-red {{ color:#DC2626; }} .txt-green {{ color:#059669; }}
  .play-grid {{ display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:6px; }}
  .play-card {{ border-radius:9px; padding:11px 12px; border:1.2px solid #E2E8F0; }}
  .p1 {{ background:#FEFCE8; border-color:#FDE047; }}
  .p2 {{ background:#FFF7ED; border-color:#FDBA74; }}
  .p3 {{ background:#FEF2F2; border-color:#FCA5A5; }}
  .play-card h3 {{ margin:0 0 6px; font-size:11.5px; color:#0F172A; }}
  .play-card ul {{ margin:0 0 0 16px; padding:0; font-size:9.2px; line-height:1.6; }}
  .chart-tip {{ background:#F8FAFC; border:1px solid #E2E8F0; border-left:4px solid #0F766E; border-radius:6px; padding:7px 10px; margin-top:8px; font-size:9.2px; color:#334155; }}
  img.chart {{ width:100%; height:auto; display:block; }}
  .page-break {{ page-break-before:always; break-before:page; }}
</style></head><body><div class="wrap">

  <!-- PAGE 1: 핵심 KPI + 30초 가이드 + 6대 위기 요약표 -->
  <div class="head">
    <h1>내 계좌 6대 위기 스트레스 테스트 · 폭락장 대응 리포트 ({as_of_str})</h1>
    <div class="sub">기준 포트폴리오: {STRESS_CONFIG['PRIMARY_NAME']} · 기준자산 {INIT_M:,}만 원 · 월 적립 {STRESS_CONFIG['MONTHLY_DCA_MAN']:,}만 원 · Yahoo 10년 실측 상관관계 반영</div>
  </div>

  <div class="kpi-row">
    <div class="kpi">
      <div class="k-lbl">최악 위기(2008급) 예상 낙폭</div>
      <div class="k-val">{pri_worst:+.1f}%</div>
      <div class="k-sub">반도체 단독({semi_worst:+.1f}%) 대비 분산 방어 확인</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">2008급 위기에도 배당 방어율</div>
      <div class="k-val">81% ~ 104% 유지</div>
      <div class="k-sub">주가가 -30% 빠져도 배당 현금흐름은 견고</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">적립+리밸런싱 원금회복 단축</div>
      <div class="k-val">{REC_REB}개월 만에 회복</div>
      <div class="k-sub">방치({REC_LUMP}개월) 대비 {rec_saved}개월 빠른 탈출</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">폭락장 리밸런싱 4년 추가수익</div>
      <div class="k-val">{diff_man:+,.0f}만 원</div>
      <div class="k-sub">단순 적립 대비 저가 매수·비중복원 보너스</div>
    </div>
  </div>

  <div class="guide-box">
    <b>📖 30초 만에 읽는 스트레스 테스트 활용법 (폭락장 예방접종!)</b>
    <div class="guide-grid">
      <div><b>1) 위기에서 달러(C층)가 에어백 역할을 합니다:</b><br>2008·2020·2022년 모두 한국 주가가 급락할 때 원/달러 환율이 급등(+10~35%)해 미국 C층이 원화 기준 낙폭을 크게 줄여주었습니다.</div>
      <div><b>2) 주가 낙폭과 배당 낙폭은 다릅니다:</b><br>우량 배당주는 주가가 30~40% 빠져도 배당금은 0~18% 수준만 변동합니다. 배당을 재투자하면 폭락장이 오히려 수량을 싸게 늘리는 기회가 됩니다.</div>
      <div><b>3) 미리 적어둔 매뉴얼이 저점 매도를 막습니다:</b><br>2페이지의 단계별 폭락장 행동 매뉴얼(-15% / -25% / -35%)을 평소에 정해두면 공포장에서도 기계적으로 대응할 수 있습니다.</div>
    </div>
  </div>

  <div class="card">
    <h2>01. 6대 역사적·가상 위기 시나리오별 내 계좌 충격 &amp; 배당 방어력 요약표</h2>
    {tbl_stress_html}
  </div>

  <!-- PAGE 2: 사전 작성 폭락장 단계별 행동 매뉴얼 카드 -->
  <div class="card page-break">
    <h2>02. 미리 정해두는 「폭락장 3단계 행동 매뉴얼」 (감정 배제 체크리스트)</h2>
    <div class="play-grid">
      <div class="play-card p1">
        <h3>🟡 1단계: 고점 대비 -15% 조정 (흔한 조정장)</h3>
        <ul>
          <li><b>매도 금지:</b> 단순 시장 변동성 구간이므로 보유 종목 매도 금지.</li>
          <li><b>신규 적립금:</b> 평소대로 월 적립 지속하되, 목표 비중(50/20/30)보다 <b>가장 덜 채워진 층</b>부터 매수.</li>
          <li><b>우선주 체크:</b> 우선주 괴리율이 10년 하위 30% 이하로 벌어졌으면 보통주 대신 우선주 적립.</li>
          <li><b>정성 점검:</b> DART 공시 감시기에서 신호 ⑤·⑦ 경고만 없는지 확인.</li>
        </ul>
      </div>
      <div class="play-card p2">
        <h3>🟠 2단계: 고점 대비 -25% 약세장 (공포 시작)</h3>
        <ul>
          <li><b>변동성 게이트 완화 확인:</b> 시장 전체 급락으로 고정 변동성 35% 컷이 막히더라도, <b>상대 70% 변동성 게이트</b> 기준으로 정상 적립 유지.</li>
          <li><b>자동 리밸런싱 발동:</b> 환율 급등으로 C층(미국) 비중이 목표(30%)보다 +5%p 이상 커졌다면, <b>신규 자금 100%를 저평가된 A·B층에 집중</b>.</li>
          <li><b>배당 재투자(DRIP):</b> 들어오는 분기·결산 배당금을 현금 방치하지 않고 저평가 우량주 매수에 즉시 재투자.</li>
        </ul>
      </div>
      <div class="play-card p3">
        <h3>🔴 3단계: 고점 대비 -35% 이상 패닉장 (기회 구간)</h3>
        <ul>
          <li><b>절대 원칙:</b> 배당 삭감(업황성)이나 자본잠식(신호 ⑦ 경고) 없는 우량주는 <b>절대 손절 금지</b>.</li>
          <li><b>적극 리밸런싱:</b> 환율 급등으로 방어된 C층 일부 이익 실현 또는 비상 현금으로 <b>할인 폭이 극대화된 A·B층 비중 복원</b>(회복 기간 최대 1년 이상 단축).</li>
          <li><b>뉴스 차단:</b> 계좌 평가손익 대신 <b>「올해 예상 연간 배당금 총액」</b>만 확인하며 10년 보유 원칙 사수.</li>
        </ul>
      </div>
    </div>
    <div class="chart-tip" style="margin-top:12px;">
      <b>📌 핵심 기억 문장:</b> <i>"폭락장에서 계좌 평가액은 일시적으로 -30% 흔들리지만, 내가 모은 우량 배당주의 <b>주식 수와 연간 배당 현금흐름은 줄어들지 않는다.</b> 하락장에서 적립을 멈추지 않고 비중만 기계적으로 복원하는 것이 10년 복리의 핵심이다."</i>
    </div>
  </div>

  <!-- PAGE 3: 차트 ① 6대 위기 비교 -->
  <div class="card page-break">
    <h2>03. [차트 ①] 6대 위기 시나리오별 포트폴리오 최대 손실폭 비교</h2>
    <img class="chart" src="data:image/png;base64,{CHART1_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 막대가 아래로 덜 내려갈수록 위기 방어력이 좋은 포트폴리오입니다.
      반도체 단독(빨강)은 2008·2022·AI 쇼크에서 -34~-50%로 크게 흔들리지만, <b>A+B+C 분산(진녹·파랑)</b>은 환율 급등 효과와 배당주 방어력 덕분에 전 구간에서 충격이 크게 완화됩니다.
    </div>
  </div>

  <!-- PAGE 4: 차트 ② 폭락장 대응 시뮬레이션 -->
  <div class="card page-break">
    <h2>04. [차트 ②] -35% 폭락장에서 대응 방식별 원금 회복 속도 &amp; 4년 자산 차이</h2>
    <img class="chart" src="data:image/png;base64,{CHART2_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 왼쪽 그래프에서 0% 점선을 먼저 뚫고 올라오는 선이 원금을 가장 빨리 회복한 전략입니다.
      폭락장에서 <b>매월 적립을 멈추지 않고(파랑) + 덜 빠진 층에서 많이 빠진 층으로 비중을 복원(진녹)</b>하면, 거치식 방치(회색)보다 원금 회복이 <b>{rec_saved}개월 빨라지고</b> 4년 뒤 자산도 크게 늘어납니다.
    </div>
  </div>

  <!-- PAGE 5: 차트 ③ 10년 실측 상관관계 히트맵 -->
  <div class="card page-break">
    <h2>05. [차트 ③] 내 포트폴리오 핵심 자산군 10년 실측 상관관계 히트맵</h2>
    <img class="chart" src="data:image/png;base64,{CHART3_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 숫자가 <b>+1.00(진한 빨강)에 가까울수록 똑같이 움직이고</b>, <b>0.00 또는 마이너스(초록)에 가까울수록 반대로 움직여 분산 효과가 크다</b>는 뜻입니다.
      가운데 <b>「원/달러 환율」</b> 행이 국내 주식(KB금융·삼성전자우 등)과 <b>마이너스(-) 상관관계</b>를 보이기 때문에, 달러 기반 미국 ETF(C층)를 30~45% 섞어두면 폭락장 방어력이 구조적으로 올라갑니다.
    </div>
  </div>

</div></body></html>"""

r_html = f"stress_test_report_{run_stamp}.html"
r_pdf  = f"stress_test_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, r_pdf, [CHART1_B64, CHART2_B64, CHART3_B64])
print(f"[리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else '건너뜀'}")

# Google Drive 자동 저장 (+ Cell 2 미연결 시 재시도 & 실패 시 브라우저 다운로드)
if IN_COLAB and STRESS_CONFIG.get("USE_GOOGLE_DRIVE", True) and DRIVE_REPORT_DIR is None:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=True)
        DRIVE_ROOT = Path(STRESS_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"[Drive 미연결] 로컬(/content/{r_pdf if pdf_ok else r_html}) 저장됨 ({type(e).__name__}: {e})")
        try:
            from google.colab import files
            files.download(r_pdf if pdf_ok else r_html)
        except Exception:
            pass

if DRIVE_REPORT_DIR is not None:
    try:
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        (DRIVE_REPORT_DIR / "stress_test_report_latest.html").write_text(html_report, encoding="utf-8")
        if pdf_ok and Path(r_pdf).exists():
            pb = Path(r_pdf).read_bytes()
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(pb)
            (DRIVE_REPORT_DIR / "stress_test_report_latest.pdf").write_bytes(pb)
        STRESS_DF.to_csv(DRIVE_REPORT_DIR / f"stress_test_summary_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        STRESS_DF.to_csv(DRIVE_REPORT_DIR / "stress_test_summary_latest.csv", index=False, encoding="utf-8-sig")
        print(f"[Drive 저장 검증 완료] {DRIVE_REPORT_DIR / 'stress_test_report_latest.pdf'}")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")